# Fine-tune Kanana Safeguard 8B on K-MHaS

Train a PEFT LoRA adapter to classify Korean news comments, inspect its learning curves, and export a merged model for [notebook 5](5_compare_guardrail_kanana_lora.ipynb). The training follows the [reference LoRA notebook](https://github.com/hyogrin/rhoai-model-training-lab/blob/main/notebooks_ko/03_lora_finetuning.ipynb).

**Run All** prepares the data, trains on one GPU and plots the results as a single OpenShift AI pipeline. To run the same three steps interactively, select `workbench` below in a CUDA PyTorch workbench with the listed training packages installed.

- Start with the repository requirements, a ready Data Science Pipelines server and an S3 bucket accessible to its launcher and your workbench. Set `NAMESPACE`, `PIPELINE_API_URL`, the S3 settings in `sample.env`, and an OpenShift token or active `oc` login. For a private pipeline CA, set `SSL_CERT_FILE` to the CA bundle.
- Allow Hugging Face and PyPI access from the pods. Budget one 40 GiB GPU for the lab, 48 GiB of host RAM and at least 45 GiB of local scratch for weights and export. Each run stores roughly 16 GB of merged weights plus a separate adapter in S3.

## 1. Choose the run size

The default uses up to 2,000 training comments and 200 development comments. Use `train_limit=0` for all eligible training rows and give each new experiment its own run label.

In [ ]:
import hashlib
import json
import os
import re
import sys
import uuid
from pathlib import Path

import yaml
from dotenv import set_key
from IPython.display import Image, display
from kfp import Client, compiler, dsl
from kfp.dsl import Artifact, Dataset, HTML, Input, Model, Output

lab_dir = Path.cwd() if (Path.cwd() / "kanana_lab.py").is_file() else Path.cwd() / "5_eval_guardrail"
sys.path.insert(0, str(lab_dir))
from kanana_lab import (
    BASE_MODEL, ENV_FILE, LAB, MODEL_REVISION, PARQUET_ROOT, PROMPT_TEMPLATE, RESULTS,
    ROOT, auth_token, load_lab_env, required, s3_client, write_json,
)

load_lab_env()
MODE = "pipeline"  # "pipeline" or "workbench"
RUN_LABEL = "kmhas-01"
if MODE not in {"pipeline", "workbench"} or not re.fullmatch(r"[a-z0-9][a-z0-9-]{0,39}", RUN_LABEL):
    raise ValueError("Choose a supported mode and a short lowercase run label.")

# Use the NeMo input rail from notebook 3, including its Korean summarization request.
prompts = yaml.safe_load((LAB / "manifests/kanana_prompts.yaml").read_text())["prompts"]
rail = next(p["content"] for p in prompts if p["task"].startswith("content_safety_check_input "))
request_template = rail.strip().replace("{{ user_input }}", PROMPT_TEMPLATE)
params = dict(
    parquet_root=PARQUET_ROOT, model_id=BASE_MODEL, revision=MODEL_REVISION,
    request_template=request_template, train_limit=2000, dev_size=200, seed=42,
    epochs=1, learning_rate=2e-4, rank=16,
)
run_dir = RESULTS / RUN_LABEL
run_dir.mkdir(parents=True, exist_ok=True)
print(f"✅ {MODE} | {RUN_LABEL} | {params['train_limit'] or 'all'} training examples | rank {params['rank']}")

## 2. Prepare disjoint training and development sets

Read the author's published Parquet files and map label 8 to `<SAFE>` and labels 0–7, including profanity, to the single hate target `<UNSAFE-S1>`. Remove documents found in the official validation split, then draw a seeded development set from the remaining training comments. [K-MHaS labels and splits](https://huggingface.co/datasets/jeanlee/kmhas_korean_hate_speech/viewer) · [Kanana taxonomy](https://huggingface.co/kakaocorp/kanana-safeguard-8b)

In [ ]:
CPU_IMAGE = "python:3.11-slim"
GPU_IMAGE = "pytorch/pytorch:2.7.1-cuda12.6-cudnn9-runtime"
DATA_PACKAGES = ["datasets==4.1.1"]
TRAIN_PACKAGES = DATA_PACKAGES + [
    "transformers==4.56.2", "trl==0.23.1", "peft==0.17.1", "accelerate==1.10.1",
]

@dsl.component(base_image=CPU_IMAGE, packages_to_install=DATA_PACKAGES)
def prepare_kmhas(parquet_root: str, train_limit: int, dev_size: int, seed: int,
                  prepared: Output[Dataset]):
    import random
    from datasets import Dataset as HFDataset, DatasetDict, load_dataset

    # Pipeline parameter values can arrive as integral JSON floats.
    train_limit, dev_size, seed = map(int, (train_limit, dev_size, seed))
    if train_limit < 0 or dev_size < 1:
        raise ValueError("train_limit must be nonnegative and dev_size must be positive.")
    files = {s: f"{parquet_root}/{s}/0000.parquet" for s in ("train", "validation")}
    source = load_dataset("parquet", data_files=files)

    def normalize(text):
        # The author release preserves some TSV quotation marks that the evaluation mirror decodes.
        return text.strip().strip('"').replace('""', '"').strip()

    held_out = {normalize(text) for text in source["validation"]["text"]}
    targets = {}
    conflicts = set()
    overlap = 0
    for row in source["train"]:
        text = normalize(row["text"])
        labels = set(row["label"])
        if not labels or not labels <= set(range(9)) or (8 in labels and labels != {8}):
            raise ValueError(f"Unexpected K-MHaS label set: {labels}")
        if text in held_out:
            overlap += 1
            continue
        if not text:
            continue
        target = "<SAFE>" if labels == {8} else "<UNSAFE-S1>"
        if targets.setdefault(text, target) != target:
            conflicts.add(text)

    rows = [{"document": text, "verdict": target} for text, target in targets.items() if text not in conflicts]
    random.Random(seed).shuffle(rows)
    if dev_size >= len(rows):
        raise ValueError(f"Reserve fewer than {len(rows)} documents for development.")
    dev_rows = rows[:dev_size]
    train_rows = rows[dev_size:][:train_limit or None]
    data = DatasetDict(train=HFDataset.from_list(train_rows), dev=HFDataset.from_list(dev_rows))
    data.save_to_disk(prepared.path)
    prepared.metadata.update({
        "train_rows": len(train_rows), "dev_rows": len(dev_rows), "source_train_rows": len(source["train"]),
        "overlap_rows_removed": overlap, "conflicting_documents_removed": len(conflicts),
    })
    print(prepared.metadata)

print("✅ Data preparation component defined")

## 3. Train and export the adapter

Apply rank-16 LoRA to the attention and MLP projections using PEFT and TRL. Only the verdict and end-of-turn tokens contribute to the loss. The adapter and the merged Transformers checkpoint are separate pipeline outputs.

In [ ]:
@dsl.component(base_image=GPU_IMAGE, packages_to_install=TRAIN_PACKAGES)
def train_lora(prepared: Input[Dataset], model_id: str, revision: str, request_template: str,
               epochs: int, learning_rate: float, rank: int, seed: int,
               adapter: Output[Model], merged: Output[Model], history: Output[Artifact]):
    import json
    import math
    from pathlib import Path
    from tempfile import TemporaryDirectory

    import torch
    from datasets import load_from_disk
    from peft import LoraConfig, TaskType, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
    from trl import SFTConfig, SFTTrainer

    epochs, rank, seed = map(int, (epochs, rank, seed))
    if min(epochs, rank, learning_rate) <= 0:
        raise ValueError("epochs, rank and learning_rate must be positive.")
    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, padding_side="right")
    tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
    verdict_ids = {}
    for verdict in ("<SAFE>", "<UNSAFE-S1>"):
        ids = tokenizer.encode(verdict, add_special_tokens=False)
        if len(ids) != 1 or tokenizer.convert_ids_to_tokens(ids[0]) != verdict:
            raise ValueError(f"The tokenizer must encode {verdict} as one verdict token.")
        verdict_ids[verdict] = ids[0]

    def encode(row):
        user_input = request_template.format(document=row["document"])
        prefix = tokenizer.apply_chat_template([{"role": "user", "content": user_input}], tokenize=True)
        ids = prefix + [verdict_ids[row["verdict"]], tokenizer.eos_token_id]
        if len(ids) > 512:
            raise ValueError("An example exceeds 512 tokens. Increase the context budget before training.")
        return {"input_ids": ids, "completion_mask": [0] * len(prefix) + [1, 1]}

    raw = load_from_disk(prepared.path)
    data = raw.map(encode, remove_columns=raw["train"].column_names)
    cuda = torch.cuda.is_available()
    if torch.version.cuda and not cuda:
        raise RuntimeError("CUDA PyTorch cannot see a GPU. Check the pod's GPU allocation and driver.")
    bf16 = cuda and torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if bf16 else (torch.float16 if cuda else torch.float32)
    base = AutoModelForCausalLM.from_pretrained(
        model_id, revision=revision, torch_dtype=dtype, attn_implementation="sdpa",
    )
    base.config.use_cache = False
    model = get_peft_model(base, LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=rank, lora_alpha=2 * rank, lora_dropout=0.05, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    ))
    model.print_trainable_parameters()
    updates_per_epoch = math.ceil(len(data["train"]) / 8)
    with TemporaryDirectory() as scratch:
        args = SFTConfig(
            output_dir=scratch, num_train_epochs=epochs, learning_rate=learning_rate,
            per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8,
            lr_scheduler_type="cosine", warmup_ratio=0.03, weight_decay=0.01,
            use_cpu=not cuda, bf16=bf16, fp16=cuda and not bf16,
            gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
            max_length=512, packing=False, completion_only_loss=True,
            dataset_kwargs={"skip_prepare_dataset": True}, report_to="none", seed=seed,
            logging_first_step=True, logging_steps=max(1, updates_per_epoch // 20),
            eval_strategy="steps", eval_steps=max(1, updates_per_epoch // 4), save_strategy="no",
        )
        trainer = SFTTrainer(model=model, args=args, processing_class=tokenizer,
                             train_dataset=data["train"], eval_dataset=data["dev"])
        trainer.train()
        final = trainer.evaluate()
        model.save_pretrained(adapter.path)
        merged_model = model.merge_and_unload()
        merged_model.config.use_cache = True
        merged_model.save_pretrained(merged.path, safe_serialization=True, max_shard_size="4GB")
        tokenizer.save_pretrained(merged.path)
        result = {
            "base_model": model_id, "revision": revision, "rank": rank, "epochs": epochs,
            "learning_rate": learning_rate, "seed": seed, "data": prepared.metadata,
            "adapter_uri": adapter.uri, "model_uri": merged.uri, "final_dev_loss": final["eval_loss"],
            "history": trainer.state.log_history,
        }
        Path(history.path).write_text(json.dumps(result), encoding="utf-8")

print("✅ LoRA training component defined")

## 4. Plot the learning curves

The left panel compares training and development loss at their recorded optimizer steps. The learning-rate schedule and gradient norms help interpret a plateau or a spike. Inspect the actual filtering decisions in notebook 5.

In [ ]:
@dsl.component(base_image=CPU_IMAGE, packages_to_install=["matplotlib==3.9.4"])
def plot_training(history: Input[Artifact], analysis: Output[Artifact], report: Output[HTML]):
    import base64
    import json
    from pathlib import Path

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MaxNLocator

    summary = json.loads(Path(history.path).read_text())
    directory = Path(analysis.path)
    directory.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), layout="constrained")
    for ax, series, label, color in [
        (axes[0], "loss", "Training", "#276788"), (axes[0], "eval_loss", "Development", "#b65520"),
        (axes[1], "learning_rate", "Learning rate", "#276788"), (axes[2], "grad_norm", "Gradient norm", "#537a46"),
    ]:
        points = {row["step"]: row[series] for row in summary["history"] if series in row}
        if not points:
            raise ValueError(f"Trainer history has no {series} values.")
        ax.plot(points.keys(), points.values(), ".-", color=color, label=label)
        ax.set_xlabel("Optimizer step")
    for ax, title in zip(axes, ["Verdict and end-of-turn loss", "Learning rate", "Gradient norm"]):
        ax.set_title(title)
        ax.grid(alpha=0.22)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    axes[0].ticklabel_format(axis="y", useOffset=False)
    axes[0].legend()
    axes[0].set_ylabel("Cross-entropy")
    fig.suptitle(f"Kanana LoRA · {summary['data']['train_rows']:,} training comments · rank {summary['rank']}")
    png = directory / "training_curves.png"
    fig.savefig(png, dpi=150)
    plt.close(fig)
    (directory / "training.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    encoded = base64.b64encode(png.read_bytes()).decode("ascii")
    Path(report.path).write_text(
        f'<html><body><img alt="Kanana training curves" style="max-width:100%" src="data:image/png;base64,{encoded}"></body></html>',
        encoding="utf-8",
    )

print("✅ Training analysis component defined")

## 5. Compile the OpenShift AI pipeline

The dataset and training log connect the three tasks. Only the training task requests a GPU. The compiled YAML can also be imported in the OpenShift AI Pipelines dashboard.

In [ ]:
@dsl.pipeline(name="kanana-kmhas-lora")
def kanana_pipeline(parquet_root: str = PARQUET_ROOT, model_id: str = BASE_MODEL,
                    revision: str = MODEL_REVISION, request_template: str = request_template,
                    train_limit: int = 2000, dev_size: int = 200, seed: int = 42,
                    epochs: int = 1, learning_rate: float = 0.0002, rank: int = 16):
    prepare = prepare_kmhas(parquet_root=parquet_root, train_limit=train_limit, dev_size=dev_size, seed=seed)
    train = train_lora(
        prepared=prepare.output, model_id=model_id, revision=revision, request_template=request_template,
        epochs=epochs, learning_rate=learning_rate, rank=rank, seed=seed,
    )
    train.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1)
    train.set_cpu_request("4").set_memory_request("32Gi").set_memory_limit("48Gi")
    plot = plot_training(history=train.outputs["history"])
    for task in (prepare, train, plot):
        task.set_env_variable("HOME", "/tmp")
        task.set_env_variable("HF_HOME", "/tmp/hf-cache")
        task.set_env_variable("MPLCONFIGDIR", "/tmp/matplotlib")

pipeline_file = run_dir / "kanana-kmhas-lora.yaml"
compiler.Compiler().compile(kanana_pipeline, str(pipeline_file))
print(f"✅ Compiled {pipeline_file}")

## 6. Execute the workflow

Pipeline mode submits one run and records its ID before waiting. Rerun these cells to reconnect after a kernel interruption. Workbench mode keeps both model directories under `workbench/` in the run directory so an interrupted S3 export can resume. Install the packages in `TRAIN_PACKAGES` from section 2 in a CUDA PyTorch kernel for that mode.

In [ ]:
from filelock import FileLock

bucket = required("AWS_S3_BUCKET")
namespace = required("NAMESPACE")
host = required("PIPELINE_API_URL") if MODE == "pipeline" else ""
s3 = s3_client()
state_path = run_dir / "run.json"
settings = {
    "params": params, "mode": MODE, "bucket": bucket, "namespace": namespace, "host": host,
    "spec_hash": hashlib.sha256(pipeline_file.read_bytes()).hexdigest(),
}
with FileLock(str(run_dir / ".execution.lock"), timeout=0):
    resuming = state_path.is_file()
    if resuming:
        state = json.loads(state_path.read_text())
        if state["settings"] != settings:
            raise ValueError("These settings differ from the saved run. Choose a new RUN_LABEL in section 1.")
    else:
        state = {"settings": settings, "prefix": f"guardrail/kanana-lora/{uuid.uuid4().hex}"}
        write_json(state_path, state)
    prefix = state["prefix"]

    if MODE == "pipeline":
        client = Client(host=host, existing_token=auth_token(), namespace=namespace,
                        ssl_ca_cert=os.getenv("SSL_CERT_FILE") or None)
        if "run_id" not in state:
            run_name = f"kanana-{RUN_LABEL}-{prefix[-10:]}"
            run = None
            if resuming:
                # A previous submission may have succeeded just before the kernel disconnected.
                match = {"predicates": [{"key": "display_name", "operation": "EQUALS", "stringValue": run_name}]}
                matches = client.list_runs(namespace=namespace, filter=json.dumps(match)).runs or []
                if matches:
                    run = matches[0]
            if run is None:
                run = client.create_run_from_pipeline_package(
                    pipeline_file=str(pipeline_file), arguments=params, experiment_name="kanana-guardrail",
                    run_name=run_name, namespace=namespace, pipeline_root=f"s3://{bucket}/{prefix}", enable_caching=False,
                )
            state["run_id"] = run.run_id
            write_json(state_path, state)
        print(f"✅ Pipeline run {state['run_id']} · OpenShift AI → Pipelines → Runs")
    elif not state.get("completed"):
        work = run_dir / "workbench"
        work.mkdir(exist_ok=True)
        prepared = Dataset(uri=str(work / "prepared"))
        adapter, merged = Model(uri=str(work / "adapter")), Model(uri=str(work / "merged"))
        history = Artifact(uri=str(work / "history.json"))
        analysis, report = Artifact(uri=str(work / "analysis")), HTML(uri=str(work / "report.html"))
        if not state.get("locally_trained"):
            import torch
            if not torch.cuda.is_available():
                raise RuntimeError("Workbench mode needs a CUDA GPU. Use pipeline mode from a CPU workbench.")
            prepare_kmhas.python_func(**{k: params[k] for k in ("parquet_root", "train_limit", "dev_size", "seed")},
                                      prepared=prepared)
            train_lora.python_func(**{k: params[k] for k in ("model_id", "revision", "request_template", "epochs",
                                                              "learning_rate", "rank", "seed")},
                                   prepared=prepared, adapter=adapter, merged=merged, history=history)
            state["locally_trained"] = True
            write_json(state_path, state)
        for name, directory in (("adapter", adapter.path), ("merged", merged.path)):
            for file in Path(directory).rglob("*"):
                if file.is_file():
                    s3.upload_file(str(file), bucket, f"{prefix}/{name}/{file.relative_to(directory).as_posix()}")
        summary = json.loads(Path(history.path).read_text())
        summary.update(adapter_uri=f"s3://{bucket}/{prefix}/adapter", model_uri=f"s3://{bucket}/{prefix}/merged")
        write_json(Path(history.path), summary)
        plot_training.python_func(history=history, analysis=analysis, report=report)
        for file in Path(analysis.path).iterdir():
            s3.upload_file(str(file), bucket, f"{prefix}/analysis/{file.name}")
        state["completed"] = True
        write_json(state_path, state)
        print("✅ Workbench training and export completed")
    else:
        print("✅ Reusing the completed workbench run")

## 7. Inspect the result and pass the model to notebook 5

Wait for the recorded run, download its graph and history, and save the exact merged-model URI. The pipeline dashboard also shows the graph in the plotting task's HTML artifact.

In [ ]:
import time
from pathlib import PurePosixPath
from urllib.parse import urlparse

if MODE == "pipeline":
    last_state = None
    while True:
        run = client.get_run(state["run_id"])
        status = str(run.state).upper()
        if status != last_state:
            print(f"{state['run_id']}: {status}")
            last_state = status
        if status == "SUCCEEDED":
            break
        if status in {"FAILED", "CANCELED", "CANCELLED", "CANCELING", "SKIPPED", "ERROR", "PAUSED"}:
            raise RuntimeError(f"Run {state['run_id']} is {status}. Inspect the run in the Pipelines dashboard.")
        time.sleep(30)

pages = s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix + "/")
summaries = [obj["Key"] for page in pages for obj in page.get("Contents", [])
             if obj["Key"].endswith("/training.json")]
if len(summaries) != 1:
    raise RuntimeError(f"Expected one training.json in s3://{bucket}/{prefix}, found {len(summaries)}.")
summary_key = PurePosixPath(summaries[0])
for filename in ("training.json", "training_curves.png"):
    s3.download_file(bucket, str(summary_key.with_name(filename)), str(run_dir / filename))
summary = json.loads((run_dir / "training.json").read_text())
model_uri = urlparse(summary["model_uri"])
if model_uri.scheme != "s3" or model_uri.netloc != bucket:
    raise ValueError(f"Unexpected merged model URI: {summary['model_uri']}")
s3.head_object(Bucket=bucket, Key=model_uri.path.strip("/") + "/config.json")
set_key(str(ENV_FILE), "KANANA_LORA_RUN", str(run_dir.relative_to(ROOT)))
display(Image(filename=str(run_dir / "training_curves.png")))
print(f"Development loss: {summary['final_dev_loss']:.4f} | {summary['data']}")
print(f"Adapter: {summary['adapter_uri']}")
print(f"✅ Ready for notebook 5: {summary['model_uri']}")